# Smoke test distribué — OmniASR CTC-1B Baoulé

Ce notebook vérifie le fine-tuning sur les deux GPU T4 avant tout entraînement long : DataLoader officiel, FSDP, FP16, activation checkpointing, 20 pas, validation, checkpoint complet et mesure de la VRAM.

Le smoke test limite les audios à 20 secondes pour isoler d'abord la compatibilité logicielle et mémoire.

## 1. Aligner PyTorch sur fairseq2

Activez **GPU T4 x2** et Internet. Si cette cellule installe PyTorch 2.8, redémarrez la session Kaggle, puis réexécutez cette cellule avant de continuer.

In [ ]:
import subprocess
import sys
import torch

torch_base = torch.__version__.split('+')[0]
cuda_variant = f"cu{(torch.version.cuda or '').replace('.', '')}"
print('Python :', sys.version.split()[0], '| torch :', torch.__version__, '| variante :', cuda_variant)
if (torch_base, cuda_variant) != ('2.8.0', 'cu128'):
    subprocess.check_call([
        sys.executable, '-m', 'pip', 'install', '--no-cache-dir', '--force-reinstall',
        'torch==2.8.0', 'torchaudio==2.8.0', 'torchvision==0.23.0',
        '--index-url', 'https://download.pytorch.org/whl/cu128',
    ])
    raise RuntimeError('PyTorch 2.8.0 installé. Redémarre maintenant la session, puis reprends à cette cellule.')
print('PyTorch est compatible ; continue avec la cellule suivante.')

## 2. Installer fairseq2 et les dépôts figés

La révision Meta est la même que celle utilisée pendant le benchmark afin de garder un environnement reproductible.

In [ ]:
from pathlib import Path
import subprocess
import sys

META_COMMIT = '81f51e224ce9e74b02cc2a3eaf21b2d91d743455'
META_REPO = Path('/kaggle/working/omnilingual-asr')
PROJECT_REPO = Path('/kaggle/working/Finetune_omniASR_CTC_1B_for_baoule')
PROJECT_URL = 'https://github.com/ngangoula-jaures/Finetune_omniASR_CTC_1B_for_baoule.git'

if not META_REPO.is_dir():
    subprocess.check_call(['git', 'clone', 'https://github.com/facebookresearch/omnilingual-asr.git', str(META_REPO)])
subprocess.check_call(['git', '-C', str(META_REPO), 'fetch', '--depth', '1', 'origin', META_COMMIT])
subprocess.check_call(['git', '-C', str(META_REPO), 'checkout', '--detach', META_COMMIT])
if not PROJECT_REPO.is_dir():
    subprocess.check_call(['git', 'clone', '--depth', '1', PROJECT_URL, str(PROJECT_REPO)])
else:
    subprocess.check_call(['git', '-C', str(PROJECT_REPO), 'pull', '--ff-only'])

subprocess.check_call([
    sys.executable, '-m', 'pip', 'install', '--no-cache-dir',
    'fairseq2[arrow]==0.6.0',
    '--extra-index-url', 'https://fair.pkg.atmeta.com/fairseq2/whl/pt2.8.0/cu128',
])
subprocess.check_call([
    sys.executable, '-m', 'pip', 'install', '--no-cache-dir',
    'datasets~=3.6.0', 'huggingface_hub>=0.32,<2', 'hf_xet',
    'pandas', 'soundfile>=0.13.1', 'pyarrow>=20,<22', 'polars>=1.29.0',
    'numba', 'librosa>=0.11.0', 'unidecode', 'uroman', 'fire', 'retrying',
])
subprocess.check_call([sys.executable, '-m', 'pip', 'install', '--no-cache-dir', '--no-deps', '-e', str(META_REPO)])
print('Installation terminée.')

In [ ]:
from importlib.metadata import version
import subprocess
import torch
import torchaudio

if torch.__version__.split('+')[0] != '2.8.0' or torchaudio.__version__.split('+')[0] != '2.8.0':
    raise RuntimeError(f'Environnement incompatible : torch={torch.__version__}, torchaudio={torchaudio.__version__}')
if not torch.cuda.is_available() or torch.cuda.device_count() != 2:
    raise RuntimeError(f'Il faut exactement deux GPU visibles ; détectés : {torch.cuda.device_count()}')
for index in range(torch.cuda.device_count()):
    props = torch.cuda.get_device_properties(index)
    print(index, props.name, round(props.total_memory / 2**30, 2), 'Gio')
print('torch :', torch.__version__, '| torchaudio :', torchaudio.__version__)
print('fairseq2 :', version('fairseq2'), '| omnilingual-asr :', version('omnilingual-asr'))
subprocess.run(['nvidia-smi'], check=True)

## 3. Télécharger le dataset Tree AI Lab

Ajoutez le même secret Kaggle `HF_TOKEN` que pour la publication. Le notebook résout et enregistre le SHA exact du dataset avant de le télécharger dans le chemin attendu par la carte OmniASR.

In [ ]:
import os
from pathlib import Path
from huggingface_hub import HfApi, snapshot_download
from kaggle_secrets import UserSecretsClient

HF_DATASET_ID = 'Tree-AI-lab/baoule-asr-dataset-mixture'
DATA_ROOT = Path('/kaggle/working/baoule_ctc1b_data')
HF_TOKEN = UserSecretsClient().get_secret('HF_TOKEN')
if not HF_TOKEN:
    raise RuntimeError('Le secret Kaggle HF_TOKEN est absent ou vide.')
os.environ['HF_HOME'] = '/kaggle/working/hf-cache'
os.environ['HF_XET_HIGH_PERFORMANCE'] = '1'
dataset_info = HfApi(token=HF_TOKEN).dataset_info(HF_DATASET_ID)
DATASET_REVISION = dataset_info.sha
if not DATASET_REVISION:
    raise RuntimeError('Impossible de résoudre la révision du dataset Hugging Face.')
print('Révision Hugging Face :', DATASET_REVISION)
snapshot_download(
    repo_id=HF_DATASET_ID, repo_type='dataset', token=HF_TOKEN,
    revision=DATASET_REVISION,
    local_dir=DATA_ROOT,
)
DATASET_VERSION = DATA_ROOT / 'baoule_mixed' / 'version=0'
STATS_PATH = DATA_ROOT / 'language_distribution_0.tsv'
ASSET_CARD = DATA_ROOT / 'assets' / 'baoule_mixed.yaml'
for required in (DATASET_VERSION, STATS_PATH, ASSET_CARD):
    if not required.exists():
        raise FileNotFoundError(required)
(Path('/kaggle/working') / 'baoule_ctc1b_dataset_revision.txt').write_text(
    f'{HF_DATASET_ID}@{DATASET_REVISION}\n', encoding='utf-8'
)
print('Dataset téléchargé :', DATASET_VERSION)
del HF_TOKEN

## 4. Préflight du DataLoader officiel

Deux itérations vérifient les partitions, le décodage FLAC, le tokenizer et le batching avant de charger le modèle 1B.

In [ ]:
from pathlib import Path
import subprocess
import sys

META_REPO = Path('/kaggle/working/omnilingual-asr')
DATA_ROOT = Path('/kaggle/working/baoule_ctc1b_data')
DATASET_VERSION = DATA_ROOT / 'baoule_mixed' / 'version=0'
STATS_PATH = DATA_ROOT / 'language_distribution_0.tsv'
for required in (META_REPO, DATASET_VERSION, STATS_PATH):
    if not required.exists():
        raise FileNotFoundError(f'Prérequis absent : {required}')

dataloader_command = [
    sys.executable, '-m', 'workflows.dataprep.dataloader_example',
    f'--dataset_path={DATASET_VERSION}', '--split=train', '--num_iterations=2',
    f'--stats_path={STATS_PATH}',
]
print('Commande :', ' '.join(map(str, dataloader_command)))
subprocess.run(dataloader_command, cwd=META_REPO, check=True)

## 5. Lancer 20 pas avec FSDP sur les deux T4

Le dossier de sortie doit être vide. Le lanceur enregistre la console, la VRAM de chaque GPU, l'utilisation GPU, le temps total, l'espace disque et les shards du checkpoint.

In [ ]:
from pathlib import Path
import shutil
import subprocess
import sys

META_REPO = Path('/kaggle/working/omnilingual-asr')
PROJECT_REPO = Path('/kaggle/working/Finetune_omniASR_CTC_1B_for_baoule')
DATA_ROOT = Path('/kaggle/working/baoule_ctc1b_data')
HF_DATASET_ID = 'Tree-AI-lab/baoule-asr-dataset-mixture'
revision_file = Path('/kaggle/working/baoule_ctc1b_dataset_revision.txt')
if not revision_file.is_file():
    raise FileNotFoundError('Révision du dataset absente : relance la cellule de téléchargement.')
revision_record = revision_file.read_text(encoding='utf-8').strip()
recorded_dataset_id, separator, DATASET_REVISION = revision_record.partition('@')
if separator != '@' or recorded_dataset_id != HF_DATASET_ID or not DATASET_REVISION:
    raise RuntimeError(f'Révision du dataset invalide : {revision_record!r}')

SMOKE_CONFIG = PROJECT_REPO / 'configs' / 'ctc_1b_smoke.yaml'
SMOKE_RUNNER = PROJECT_REPO / 'src' / 'run_smoke_test.py'
SMOKE_OUTPUT = Path('/kaggle/working/omniASR-CTC-1B-baoule-smoke')
if SMOKE_OUTPUT.exists() and any(SMOKE_OUTPUT.iterdir()):
    raise FileExistsError(f'{SMOKE_OUTPUT} existe déjà. Utilise un nouveau chemin ou sauvegarde d’abord l’ancien essai.')
config_text = SMOKE_CONFIG.read_text(encoding='utf-8')
required_config_values = [
    'name: "omniASR_CTC_1B"', 'data_parallelism: "fsdp"',
    'dtype: "torch.float16"', 'num_steps: 20',
    'max_audio_len: 320_000', str(DATA_ROOT / 'assets'),
]
missing = [value for value in required_config_values if value not in config_text]
if missing:
    raise RuntimeError(f'Configuration smoke périmée ; valeurs absentes : {missing}')
free_gib = shutil.disk_usage('/kaggle/working').free / 2**30
print(f'Espace libre avant le smoke test : {free_gib:.2f} Gio')
if free_gib < 18:
    raise RuntimeError('Moins de 18 Gio libres : espace insuffisant pour le modèle et un checkpoint complet.')
smoke_command = [
    sys.executable, str(SMOKE_RUNNER),
    '--meta-repo', str(META_REPO), '--config', str(SMOKE_CONFIG),
    '--output-dir', str(SMOKE_OUTPUT), '--num-gpus', '2', '--expected-steps', '20',
    '--dataset-id', HF_DATASET_ID, '--dataset-revision', DATASET_REVISION,
]
print('Commande :', ' '.join(map(str, smoke_command)))
subprocess.run(smoke_command, check=True)

## 6. Vérifier le résultat

Le smoke test n'est validé que si le processus se termine sans erreur et produit au pas 20 les états complets du trainer, du modèle, de l'optimiseur et du DataLoader sur les deux processus. Les estimations de durée incluent le chargement et la validation : elles sont volontairement prudentes.

In [ ]:
from pathlib import Path
import json

SMOKE_OUTPUT = Path('/kaggle/working/omniASR-CTC-1B-baoule-smoke')

summary = json.loads((SMOKE_OUTPUT / 'smoke_summary.json').read_text(encoding='utf-8'))
print(json.dumps(summary, ensure_ascii=False, indent=2))
if summary['return_code'] != 0:
    raise RuntimeError('Le smoke test a échoué. Consulte smoke_console.log.')
step_20 = [item for item in summary['checkpoints'] if item['step'] == 20]
if not step_20 or not step_20[0]['complete']:
    raise RuntimeError(f"Checkpoint complet du pas 20 absent : {summary['checkpoints']}")
log_lines = (SMOKE_OUTPUT / 'smoke_console.log').read_text(encoding='utf-8', errors='replace').splitlines()
print('\n'.join(log_lines[-120:]))
print('Smoke test CTC-1B validé.')

## 7. Sauvegarde

Utilisez **Save Version** avec les sorties activées. Ne lancez pas encore les 500 premiers pas : les mesures du smoke test doivent d'abord servir à confirmer la durée maximale, l'accumulation de gradients et l'intervalle 250/500 pas.